In [ ]:
# Timer값이 클수록 중요도가 높도록 weight

import pandas as pd
import xgboost as xgb
from pathlib import Path
import re

# Step 1: 데이터 로드
csv_filename = 'sample_data_7-20_1344_largestcontentfulpaint_iter5_win1_inter5.csv'

# 운영용 FTP 다운로드 (노트북 테스트 중에는 실행하지 않음)
# from ftplib import FTP
# import io
# print("[Step 1] Connecting to FTP server...")
# ftp = FTP('feo378697.ftp.upload.akamai.com')
# ftp.login(user=os.getenv('FTP_USER', ''), passwd=os.getenv('FTP_PASSWORD', ''))
# file_data = io.BytesIO()
# ftp.cwd('378697')
# ftp.retrbinary(f'RETR {csv_filename}', file_data.write)
# ftp.quit()

# 테스트용: 프로젝트의 processed 폴더에서 CSV를 읽음
# Jupyter 실행 위치에 따라 Path.cwd()가 달라질 수 있어 설정과 동일한 절대경로를 사용
processed_dir = Path('/opt/perf-analytics/processed')
csv_path = processed_dir / csv_filename
if not csv_path.is_file():
    raise FileNotFoundError(f"Processed CSV file not found: {csv_path}")

# 파일명 패턴: sample_data_[date]_[time]_[timer_name]_iter5_win1_inter5.csv
def extract_timer_name_from_filename(filename):
    m = re.match(r'^sample_data_[^_]+_[^_]+_(.+?)_iter5_win1_inter5\.csv$', filename)
    return m.group(1) if m else 'timer'

timer_metric_name_from_file = extract_timer_name_from_filename(csv_filename)
print(f"[Step 1] Timer metric inferred from filename: {timer_metric_name_from_file}")

print(f"[Step 1] Loading CSV file from processed folder: {csv_path}")
df = pd.read_csv(csv_path)
print("[Step 1] Data was loaded successfully.")


In [ ]:
# Step 2: 범주형 데이터 처리 및 Feature 선택
print("[Step 2] Processing categorical data and selecting features...")

# deviceMemory를 categorical로 변환 (이산 값으로 취급)
df['deviceMemory'] = df['deviceMemory'].astype(str)

features_to_drop = list(set([
    'label', 'timer', 'cacherate', 'transferbyte', 'bodysize',
    'requestcount', 'cdncacherate', 'edgetime', 'origintime', 'rtt'
]))

print(f"최종적으로 제외될 feature: {features_to_drop}")

X = df.drop(features_to_drop, axis=1)
y = df['label']

# Convert categorical columns to numeric using one-hot encoding
categorical_columns = list(X.select_dtypes(include=['object']).columns)
X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

# timer 기반 가중치 생성 (anomaly-centric)
# - anomaly(y=1) 구간의 느린 샘플에만 완만한 weight를 부여
# - sqrt로 증가폭을 완만하게, clip([1, 5])으로 outlier 과대반영 방지
# - normal(y=0)은 weight=1로 유지하여 분류 성능(PR-AUC) 보존
weights = pd.Series(1.0, index=df.index)
anomaly_mask = (y == 1)
anomaly_mean = df.loc[anomaly_mask, 'timer'].mean() if anomaly_mask.any() else df['timer'].mean()
anomaly_weight = (df.loc[anomaly_mask, 'timer'] / anomaly_mean) ** 0.5
anomaly_weight = anomaly_weight.clip(1.0, 5.0)
weights[anomaly_mask] = anomaly_weight

In [ ]:
# Step 3: 모델 학습
print("[Step 3] Training the XGBoost model...")
model = xgb.XGBClassifier(eval_metric='logloss')
model.fit(X, y, sample_weight=weights)

In [ ]:
# Step 4: 모델 학습
print("[Step 4] Explaining model output...")
import shap
import numpy as np

# SHAP Explainer 생성
explainer = shap.Explainer(model)
shap_values = explainer(X)

# 가장 영향력 있는 Feature 시각화 (Beeswarm plot)
shap.plots.beeswarm(shap_values)

# 상위 원인 텍스트로 추출
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': np.abs(shap_values.values).mean(0)
}).sort_values(by='importance', ascending=False)

# 중요도 정규화
# feature_importance['normalized_importance'] = feature_importance['importance'] / feature_importance['importance'].sum()

print("--- 상위 10개 원인 (정규화된 중요도) ---")
print(feature_importance.head(10))

In [ ]:
# Dependence Plot 추가
# print("[Step 4] Generating dependence plot for bodysize...")
# shap.plots.scatter(shap_values[:, 'bodysize'], color=shap_values)

In [ ]:
# Step 5-A: Prepare Direction-Aware Analysis Context
import os
import numpy as np
import pandas as pd

print("[Step 5-A] Preparing analysis context...")

ollama_model = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")
timer_metric_name = os.getenv("TIMER_METRIC_NAME", globals().get('timer_metric_name_from_file', "timer"))
anomaly_threshold_desc = os.getenv("ANOMALY_THRESHOLD_DESC", "unknown statistical threshold")


def _load_conf(path):
    conf = {}
    try:
        with open(path) as f:
            for raw in f:
                line = raw.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" in line:
                    k, _, v = line.partition("=")
                    conf[k.strip()] = v.strip()
    except FileNotFoundError:
        print(f"[Warning] Config file not found: {path}")
    return conf


def _compute_timer_state(df, y, timer_metric_name):
    raw_timer_p75_all = float(df['timer'].quantile(0.75))
    raw_timer_p75_anomaly = float(df.loc[y == 1, 'timer'].quantile(0.75)) if (y == 1).any() else np.nan
    raw_timer_p75_normal = float(df.loc[y == 0, 'timer'].quantile(0.75)) if (y == 0).any() else np.nan

    normal_timer_ms = int(round(raw_timer_p75_normal)) if not np.isnan(raw_timer_p75_normal) else None
    anomaly_timer_ms = int(round(raw_timer_p75_anomaly)) if not np.isnan(raw_timer_p75_anomaly) else None
    timer_transition_label = (
        f"{normal_timer_ms}(ms) -> {anomaly_timer_ms}(ms)"
        if normal_timer_ms is not None and anomaly_timer_ms is not None
        else "N/A(ms) -> N/A(ms)"
    )

    if np.isnan(raw_timer_p75_anomaly) or np.isnan(raw_timer_p75_normal):
        anomaly_direction = "worsening"
    else:
        anomaly_direction = "worsening" if raw_timer_p75_anomaly >= raw_timer_p75_normal else "improving"

    if np.isnan(raw_timer_p75_anomaly) or np.isnan(raw_timer_p75_normal):
        direction_en = "worsened"
        timer_change_summary = f"Unable to compute a reliable normal-vs-anomaly delta for {timer_metric_name}."
        plain_transition_summary = f"The normal-vs-anomaly transition for {timer_metric_name} is unavailable."
    else:
        timer_delta_sec = raw_timer_p75_anomaly - raw_timer_p75_normal
        direction_en = "worsened" if timer_delta_sec >= 0 else "improved"
        timer_change_summary = (
            f"{timer_metric_name} moved from normal {raw_timer_p75_normal:.2f}ms "
            f"to anomaly {raw_timer_p75_anomaly:.2f}ms ({timer_delta_sec:+.2f}ms), indicating a sudden {direction_en} state."
        )
        plain_transition_summary = f"{timer_metric_name} changed to {timer_transition_label}, indicating a sudden {direction_en} state."

    return {
        'raw_timer_p75_all': raw_timer_p75_all,
        'raw_timer_p75_anomaly': raw_timer_p75_anomaly,
        'raw_timer_p75_normal': raw_timer_p75_normal,
        'timer_transition_label': timer_transition_label,
        'anomaly_direction': anomaly_direction,
        'timer_change_summary': timer_change_summary,
        'plain_transition_summary': plain_transition_summary,
    }


def _build_feature_impact(X, shap_array, y):
    x_matrix = X.to_numpy()
    positive_impact, negative_impact, median_impact, on_ratio, is_binary = [], [], [], [], []
    signed_mean_impact, anomaly_signed_impact, normal_signed_impact = [], [], []
    binary_on_anomaly_shap, binary_off_anomaly_shap, binary_presence_effect = [], [], []

    y_array = np.asarray(y)
    anomaly_mask = y_array == 1
    normal_mask = y_array == 0

    for feature_idx in range(shap_array.shape[1]):
        feature_shap = shap_array[:, feature_idx]
        feature_vals = x_matrix[:, feature_idx]

        unique_vals = np.unique(feature_vals)
        binary_flag = len(unique_vals) <= 2 and np.isin(unique_vals, [0, 1]).all()
        is_binary.append(binary_flag)
        on_ratio.append(float((feature_vals == 1).mean()) if binary_flag else np.nan)

        non_zero_shap = feature_shap[feature_shap != 0]
        if len(non_zero_shap) > 0:
            pos_shap = non_zero_shap[non_zero_shap > 0]
            neg_shap = non_zero_shap[non_zero_shap < 0]
            positive_impact.append(pos_shap.mean() if len(pos_shap) > 0 else 0)
            negative_impact.append(neg_shap.mean() if len(neg_shap) > 0 else 0)
            median_impact.append(np.median(non_zero_shap))
        else:
            positive_impact.append(0)
            negative_impact.append(0)
            median_impact.append(0)

        signed_mean_impact.append(float(feature_shap.mean()))
        anomaly_mean = float(feature_shap[anomaly_mask].mean()) if anomaly_mask.any() else 0.0
        normal_mean = float(feature_shap[normal_mask].mean()) if normal_mask.any() else 0.0
        anomaly_signed_impact.append(anomaly_mean)
        normal_signed_impact.append(normal_mean)

        if binary_flag and anomaly_mask.any():
            anom_on_mask = anomaly_mask & (feature_vals == 1)
            anom_off_mask = anomaly_mask & (feature_vals == 0)
            on_shap = float(feature_shap[anom_on_mask].mean()) if anom_on_mask.any() else 0.0
            off_shap = float(feature_shap[anom_off_mask].mean()) if anom_off_mask.any() else 0.0
            binary_on_anomaly_shap.append(on_shap)
            binary_off_anomaly_shap.append(off_shap)
            binary_presence_effect.append(on_shap - off_shap)
        else:
            binary_on_anomaly_shap.append(np.nan)
            binary_off_anomaly_shap.append(np.nan)
            binary_presence_effect.append(np.nan)

    feature_impact = pd.DataFrame({
        'feature': X.columns,
        'importance': np.abs(shap_array).mean(0),
        'positive_impact': positive_impact,
        'negative_impact': negative_impact,
        'median_impact': median_impact,
        'is_binary': is_binary,
        'on_ratio': on_ratio,
        'signed_mean_impact': signed_mean_impact,
        'anomaly_signed_impact': anomaly_signed_impact,
        'normal_signed_impact': normal_signed_impact,
        'binary_on_anomaly_shap': binary_on_anomaly_shap,
        'binary_off_anomaly_shap': binary_off_anomaly_shap,
        'binary_presence_effect': binary_presence_effect,
    })
    feature_impact['net_impact'] = feature_impact['positive_impact'] + feature_impact['negative_impact']
    feature_impact['anomaly_delta_impact'] = feature_impact['anomaly_signed_impact'] - feature_impact['normal_signed_impact']
    feature_impact['effect_for_ranking'] = np.where(
        feature_impact['is_binary'],
        feature_impact['binary_presence_effect'].fillna(feature_impact['anomaly_signed_impact']),
        feature_impact['anomaly_signed_impact'],
    )
    feature_impact['dominant_direction'] = np.where(
        feature_impact['effect_for_ranking'] > 0,
        'worsening',
        np.where(feature_impact['effect_for_ranking'] < 0, 'improving', 'mixed'),
    )
    return feature_impact.sort_values(by='importance', ascending=False)


def _select_directional_features(feature_impact, anomaly_direction, min_on_ratio):
    support_factor = np.where(
        feature_impact['is_binary'],
        np.clip(feature_impact['on_ratio'].fillna(0), min_on_ratio, 1.0),
        1.0,
    )

    # Reduce over-penalization on sparse one-hot features while still controlling noise.
    support_weight = np.power(support_factor, 0.25)

    feature_impact['worsening_score'] = (
        feature_impact['importance']
        * np.maximum(feature_impact['effect_for_ranking'], 0)
        * support_weight
    )
    feature_impact['improving_score'] = (
        feature_impact['importance']
        * np.abs(np.minimum(feature_impact['effect_for_ranking'], 0))
        * support_weight
    )

    worsening_candidates = feature_impact[
        feature_impact['effect_for_ranking'] > 0
    ].copy()
    worsening_candidates = worsening_candidates[
        (~worsening_candidates['is_binary']) | (worsening_candidates['on_ratio'] >= min_on_ratio)
    ]
    worsening_features = worsening_candidates.sort_values(by=['worsening_score', 'importance'], ascending=False).head(10).copy()
    worsening_features['impact_type'] = 'Performance-Worsening'

    rare_worsening = feature_impact[
        (feature_impact['effect_for_ranking'] > 0)
        & (feature_impact['is_binary'])
        & (feature_impact['on_ratio'] < min_on_ratio)
    ].sort_values(by=['effect_for_ranking', 'importance'], ascending=False).head(10).copy()

    improving_candidates = feature_impact[
        feature_impact['effect_for_ranking'] < 0
    ].copy()
    improving_candidates = improving_candidates[
        (~improving_candidates['is_binary']) | (improving_candidates['on_ratio'] >= min_on_ratio)
    ]
    improving_features = improving_candidates.sort_values(by=['improving_score', 'importance'], ascending=False).head(10).copy()
    improving_features['impact_type'] = 'Performance-Improving'

    rare_improving = feature_impact[
        (feature_impact['effect_for_ranking'] < 0)
        & (feature_impact['is_binary'])
        & (feature_impact['on_ratio'] < min_on_ratio)
    ].sort_values(by=['effect_for_ranking', 'importance'], ascending=True).head(10).copy()

    if anomaly_direction == "worsening":
        return worsening_features, rare_worsening, 'worsening_score', 'effect_for_ranking', 'Performance-Worsening'
    return improving_features, rare_improving, 'improving_score', 'effect_for_ranking', 'Performance-Improving'


def _build_feature_context_df(X, shap_array, selected_features):
    feature_context_rows = []
    for feature in selected_features['feature']:
        idx = X.columns.get_loc(feature)
        vals = X[feature].to_numpy()
        shap_col = shap_array[:, idx]

        unique_vals = np.unique(vals)
        current_binary = len(unique_vals) <= 2 and np.isin(unique_vals, [0, 1]).all()

        if current_binary:
            on_mask = vals == 1
            off_mask = vals == 0
            feature_context_rows.append({
                'feature': feature,
                'feature_type': 'binary(one-hot)',
                'on_ratio': float(on_mask.mean()),
                'on_mean_shap': float(shap_col[on_mask].mean()) if on_mask.any() else 0.0,
                'off_mean_shap': float(shap_col[off_mask].mean()) if off_mask.any() else 0.0,
                'value_shap_corr': np.nan,
                'p50_value': np.nan,
                'p90_value': np.nan,
                'p99_value': np.nan,
            })
        else:
            corr = float(np.corrcoef(vals, shap_col)[0, 1]) if np.std(vals) > 0 and np.std(shap_col) > 0 else 0.0
            feature_context_rows.append({
                'feature': feature,
                'feature_type': 'numeric/continuous',
                'on_ratio': np.nan,
                'on_mean_shap': np.nan,
                'off_mean_shap': np.nan,
                'value_shap_corr': corr,
                'p50_value': float(np.percentile(vals, 50)),
                'p90_value': float(np.percentile(vals, 90)),
                'p99_value': float(np.percentile(vals, 99)),
            })
    return pd.DataFrame(feature_context_rows)


def _check_top5_coverage(analysis_report, selected_features):
    top5_features = selected_features['feature'].head(5).tolist()
    missing_top5 = []
    for feat in top5_features:
        tokens = [feat]
        if feat.startswith("url_"):
            tokens.append(feat[len("url_"):])
        elif feat.startswith("referrer_"):
            tokens.append(feat[len("referrer_"):])
        if not any(token in analysis_report for token in tokens):
            missing_top5.append(feat)
    return missing_top5


def _build_report_prompt(
    plain_transition_summary,
    anomaly_direction,
    direction_logic_desc,
    timer_metric_name,
    anomaly_threshold_desc,
    dataset_rows,
    feature_count,
    anomaly_rate,
    selected_top5_text,
    allowed_feature_list_text,
    selected_text,
    selected_appendix_text,
    feature_context_text,
):
    return f"""You are a website performance analyst.
Your task is to explain only the root causes for the selected anomaly direction.

Language policy (strict):
- Write the entire report in English only.
- Do not use Korean, Chinese, Japanese, or mixed-language output.
- Do not copy or quote any example sentence template from this prompt.

## Report Opening Requirement (must follow)
- In section 1 (Executive Summary), first sentence must use this transition summary in your own wording:
  {plain_transition_summary}
- In section 1, add one additional sentence that summarizes the most likely user-facing cause.
- Do not output any template-style sentence verbatim.

## Direction Decision (must follow strictly)
- Selected anomaly direction: {anomaly_direction}
- Decision rule: {direction_logic_desc}
- If selected direction is worsening: explain only worsening factors.
- If selected direction is improving: explain only improving factors.
- Never describe the opposite direction in this report.

## Ground Truth Context
- Timer metric: {timer_metric_name}
- Timer semantics: Higher timer means worse page load performance.
- Anomaly threshold details: {anomaly_threshold_desc}
- Label semantics: y=1 anomaly, y=0 normal.

## Model & Data Context
- Model: XGBoost classifier with sample weights.
- Weighting rule: anomaly(y=1) samples weighted by sqrt(timer / mean_anomaly_timer), clipped to [1, 5]; normal(y=0) samples keep weight = 1.
- Dataset rows: {dataset_rows}
- Feature count after one-hot encoding: {feature_count}
- Observed anomaly rate: {anomaly_rate:.4f}

## Writing Rules For End Users (must follow)
1) Do not use technical terms such as SHAP, feature importance, one-hot encoding, percentile, score, model, or classifier.
2) Do not include numeric analysis values in the narrative body.
3) Use user-friendly expressions like page section, network condition, mobile users, and traffic pattern.
4) Explain causes and actions in practical, non-technical language.

## Mandatory Requirements
1) Mention only feature names that exist in the Allowed Feature Names list.
2) If a feature is not in the list, do not invent or guess a new name.
3) In Root Causes, explicitly cover Selected Top5 first before adding any other details.
4) If any Selected Top5 is excluded from Root Causes, explain the reason in section 7.

## Selected Top5
{selected_top5_text}

## Allowed Feature Names
{allowed_feature_list_text}

## Directional Feature Data
{selected_text}

## Appendix (rare one-hot)
{selected_appendix_text}

## Feature-wise Context
{feature_context_text}

## Required output format (Markdown)
1. Executive Summary
2. Direction-Matched Root Causes (Top drivers)
3. Business Implications
4. Actionable Recommendations
5. Monitoring Priorities
6. Interpretation Guardrails
7. Selected Top5 Exclusion Reason

## Section 7 Requirement (MANDATORY)
REGARDLESS of whether selected Top5 is covered or missing:
- If ALL selected Top5 are covered in Root Causes: Write a brief statement confirming full coverage.
- If ANY selected Top5 is missing: Explain why those features were excluded and why they are less relevant than the covered ones.
Always write Section 7 - it must appear in every report.

Keep the report concise and practical for performance engineers.
"""


def md_to_simple_html(text):
    import re

    lines = text.splitlines()
    html_lines = []
    for line in lines:
        if line.startswith("### "):
            html_lines.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            html_lines.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            html_lines.append(f"<h1>{line[2:]}</h1>")
        elif line.startswith("- "):
            html_lines.append(f"<li>{line[2:]}</li>")
        elif line.strip() == "":
            html_lines.append("<br>")
        else:
            line = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', line)
            line = re.sub(r'`([^`]+)`', r'<code>\1</code>', line)
            html_lines.append(f"<p>{line}</p>")
    return "<html><body style='font-family:Arial,sans-serif;'>" + "\n".join(html_lines) + "</body></html>"


_sec = _load_conf("/opt/perf-analytics/.sec/aws-ses")
_mail = _load_conf("/opt/perf-analytics/config/ses_email.conf")

aws_access_key_id = _sec.get("AWS_ACCESS_KEY_ID")
aws_secret_access_key = _sec.get("AWS_SECRET_ACCESS_KEY")
ses_region = _mail.get("SES_REGION", "ap-northeast-1")
ses_from_email = _mail.get("SES_FROM_EMAIL", "")
ses_to_email = _mail.get("SES_TO_EMAIL", "")

if not aws_access_key_id or not aws_secret_access_key:
    raise ValueError("AWS credentials not found in /opt/perf-analytics/.sec/aws-ses")
if not ses_from_email or not ses_to_email:
    raise ValueError("SES_FROM_EMAIL / SES_TO_EMAIL not set in /opt/perf-analytics/config/ses_email.conf")

min_on_ratio = float(os.getenv("MIN_ONEHOT_ON_RATIO", "0.01"))

timer_state = _compute_timer_state(df, y, timer_metric_name)
raw_timer_p75_all = timer_state['raw_timer_p75_all']
raw_timer_p75_anomaly = timer_state['raw_timer_p75_anomaly']
raw_timer_p75_normal = timer_state['raw_timer_p75_normal']
timer_transition_label = timer_state['timer_transition_label']
anomaly_direction = timer_state['anomaly_direction']
timer_change_summary = timer_state['timer_change_summary']
plain_transition_summary = timer_state['plain_transition_summary']

print(f"[Step 5-A] Timer change summary: {timer_change_summary}")
print(
    f"[Step 5-A] Direction decision by timer p75: "
    f"anomaly={raw_timer_p75_anomaly:.4f}, normal={raw_timer_p75_normal:.4f}, all={raw_timer_p75_all:.4f} "
    f"=> {anomaly_direction}"
)

shap_array = shap_values.values if hasattr(shap_values, 'values') else np.asarray(shap_values)
feature_impact = _build_feature_impact(X, shap_array, y)

selected_features, selected_appendix, selected_score_col, selected_effect_col, selected_type_title = _select_directional_features(
    feature_impact, anomaly_direction, min_on_ratio
)

# Keep compatibility for downstream diagnostics
worsening_candidates = feature_impact[feature_impact['effect_for_ranking'] > 0].copy()
worsening_candidates = worsening_candidates[(~worsening_candidates['is_binary']) | (worsening_candidates['on_ratio'] >= min_on_ratio)]
worsening_features = worsening_candidates.sort_values(by=['worsening_score', 'importance'], ascending=False).head(10).copy()
worsening_features['impact_type'] = 'Performance-Worsening'

improving_candidates = feature_impact[feature_impact['effect_for_ranking'] < 0].copy()
improving_candidates = improving_candidates[(~improving_candidates['is_binary']) | (improving_candidates['on_ratio'] >= min_on_ratio)]
improving_features = improving_candidates.sort_values(by=['improving_score', 'importance'], ascending=False).head(10).copy()
improving_features['impact_type'] = 'Performance-Improving'

print(f"--- Top 10 {selected_type_title.lower()} features (direction-aware) ---")
if len(selected_features) > 0:
    print(selected_features[['feature', 'importance', selected_effect_col, selected_score_col, 'on_ratio']].to_string(index=False))
else:
    print("No direction-matched features found.")

feature_context_df = _build_feature_context_df(X, shap_array, selected_features)
feature_context_text = feature_context_df.to_string(index=False) if len(feature_context_df) > 0 else "No context rows available."

selected_text = selected_features[[
    'feature', 'importance', 'positive_impact', 'negative_impact',
    'signed_mean_impact', 'anomaly_signed_impact', 'normal_signed_impact',
    'binary_on_anomaly_shap', 'binary_off_anomaly_shap', 'binary_presence_effect',
    'effect_for_ranking', 'anomaly_delta_impact', 'median_impact', 'net_impact', selected_score_col, 'on_ratio', 'impact_type'
]].to_string(index=False) if len(selected_features) > 0 else "No direction-matched features available."

selected_appendix_text = selected_appendix[[
    'feature', 'importance', 'effect_for_ranking', 'anomaly_signed_impact', 'binary_presence_effect', 'anomaly_delta_impact', 'on_ratio',
]].to_string(index=False) if len(selected_appendix) > 0 else "No rare one-hot features for this direction."

selected_top5 = selected_features.head(5).copy()
selected_top5_text = selected_top5[[
    'feature', 'importance', selected_score_col, 'on_ratio',
]].to_string(index=False) if len(selected_top5) > 0 else "No selected Top5 features."

allowed_feature_names = sorted(
    set(selected_features['feature'].tolist()) | set(selected_appendix['feature'].tolist())
)
allowed_feature_list_text = "\n".join([f"- {name}" for name in allowed_feature_names]) if allowed_feature_names else "- none"

dataset_rows = int(len(X))
feature_count = int(X.shape[1])
anomaly_rate = float(y.mean()) if len(y) > 0 else float('nan')

direction_logic_desc = (
    f"raw {timer_metric_name} p75 comparison: anomaly={raw_timer_p75_anomaly:.4f}, "
    f"normal={raw_timer_p75_normal:.4f}, all={raw_timer_p75_all:.4f}"
)

prompt = _build_report_prompt(
    plain_transition_summary=plain_transition_summary,
    anomaly_direction=anomaly_direction,
    direction_logic_desc=direction_logic_desc,
    timer_metric_name=timer_metric_name,
    anomaly_threshold_desc=anomaly_threshold_desc,
    dataset_rows=dataset_rows,
    feature_count=feature_count,
    anomaly_rate=anomaly_rate,
    selected_top5_text=selected_top5_text,
    allowed_feature_list_text=allowed_feature_list_text,
    selected_text=selected_text,
    selected_appendix_text=selected_appendix_text,
    feature_context_text=feature_context_text,
)

analysis_report = ""
missing_top5 = []
top5_coverage_note = "(Section 7 generated by first LLM call)"

In [ ]:
# Step 5-B: Generate LLM Report (English Only)
import json
import requests

def _enforce_executive_summary_ground_truth(report_text, ground_truth_line):
    lines = report_text.splitlines()
    summary_heading_idx = None
    for i, raw in enumerate(lines):
        normalized = raw.strip().lower()
        if normalized in ("### executive summary", "## executive summary", "# executive summary", "1. executive summary"):
            summary_heading_idx = i
            break

    if summary_heading_idx is None:
        return report_text

    for j in range(summary_heading_idx + 1, len(lines)):
        current = lines[j].strip()
        if not current:
            continue
        if current.startswith("#"):
            break
        lines[j] = ground_truth_line
        return "\n".join(lines)

    lines.insert(summary_heading_idx + 1, ground_truth_line)
    return "\n".join(lines)

print("[Step 5-B] Generating report from Ollama...")

try:
    response = requests.post(
        'http://localhost:11434/api/generate',
        json={
            'model': ollama_model,
            'prompt': prompt,
            'stream': True,
            'options': {'temperature': 0},
        },
        stream=True,
        timeout=300,
    )

    if response.status_code == 200:
        print("\n=== AI Analysis Report ===\n")
        for line in response.iter_lines():
            if line:
                data = json.loads(line)
                if 'response' in data:
                    chunk = data['response']
                    print(chunk, end='', flush=True)
                    analysis_report += chunk
        print("\n")

        if not analysis_report.strip():
            print("[Warning] LLM response is empty.")
        else:
            analysis_report = _enforce_executive_summary_ground_truth(
                analysis_report,
                timer_change_summary,
            )
            print("[Step 5-B] Executive Summary first sentence aligned to timer p75 ground truth.")
    else:
        print(f"Error: {response.status_code}")
        print("Please ensure Ollama server is running: ollama serve")

except requests.exceptions.ConnectionError:
    print("Error: Cannot connect to Ollama server.")
    print("Please run this command in terminal: ollama serve")
except requests.exceptions.Timeout:
    print("Error: Request timed out. The model is taking too long to respond.")
    print("Try using a faster model or reduce the prompt size.")
except requests.exceptions.HTTPError as exc:
    print(f"Error: Failed to generate report: {exc}")

In [ ]:
# Step 5-C: Build and Send Email (notebook test mode)
import boto3
from botocore.exceptions import ClientError

print("[Step 5-C] Building email payload and sending from notebook test mode...")

analysis_report_local = globals().get('analysis_report', "")
if analysis_report_local == "":
    raise RuntimeError("analysis_report is empty. Run Cell 7 first.")

subject_timer_metric_name = globals().get('timer_metric_name_from_file', None)
if not subject_timer_metric_name or subject_timer_metric_name == "timer":
    subject_timer_metric_name = extract_timer_name_from_filename(csv_filename)

# Use timer p75 ground-truth summary to keep numeric consistency.
email_summary_line = timer_change_summary

test_recipient_email = "hyoon@akamai.com"
print(f"[Step 5-C] Notebook test recipient locked to: {test_recipient_email}")

email_subject = f"[Alert Report] {subject_timer_metric_name} | {timer_transition_label}"
print(f"[Step 5-C] Email subject preview: {email_subject}")
print(f"[Step 5-C] Email summary source (ground truth): {email_summary_line}")

email_plain = "\n".join([
    "## Timer Change Summary",
    "",
    email_summary_line,
    "",
    "## AI Analysis",
    "",
    analysis_report_local,
    "",
    "### Selected Top5 Exclusion Reason",
    "",
    top5_coverage_note,
])
email_html = md_to_simple_html(email_plain)

ses_client = boto3.client(
    'sesv2',
    region_name=ses_region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)

try:
    ses_response = ses_client.send_email(
        FromEmailAddress=ses_from_email,
        Destination={'ToAddresses': [test_recipient_email]},
        Content={
            'Simple': {
                'Subject': {'Data': email_subject, 'Charset': 'UTF-8'},
                'Body': {
                    'Text': {'Data': email_plain, 'Charset': 'UTF-8'},
                    'Html': {'Data': email_html, 'Charset': 'UTF-8'},
                },
            }
        },
    )
    print(f"✅ Report email sent! MessageId: {ses_response['MessageId']}")
    print(f"   From: {ses_from_email}  To: {test_recipient_email}")
except ClientError as e:
    print(f"❌ SES send failed: {e.response['Error']['Message']}")

print("[Step 5-C] SES send attempt finished.")